# 06 · Imputation & anomaly detection

**Questions:** can we predict a chemist's missing country or specialty from other features? Which dates are probably wrong?

**Method** (`chemkg.imputation`)
- **Country:** target is the single modern country (people with exactly one affiliation), top 20 + OTHER. Features: character 2–4-grams of the name, birth decade, occupations, sex, birthplace country, words from employer/alma mater/award labels. Multinomial logistic regression (balanced class weights), stratified 5-fold CV.
  Baselines: majority class, and "birthplace country if known, else majority".
- **Specialty:** primary specialist domain, with occupation features removed (they *are* the label).
- **Dates:** rule flags, plus a relational check from the advisor graph (advisor born < 15 or > 80 years before the student).

**Assumption:** predictions are stored separately (`data/processed/imputed_*.parquet`) and **never** fed back into other analyses.

In [1]:
import json, warnings
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from chemkg.data import load, analysable
from chemkg import countries
warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 70)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "font.size": 10})
ds = load()
P, A = ds.people, analysable(ds.people)
NAMES = countries.load_iso3().set_index("iso3").name.to_dict()
print(f"{len(P):,} people, {len(A):,} analysable; enrichment loaded: {ds.enriched}")

11,903 people, 11,860 analysable; enrichment loaded: True


In [2]:
from chemkg import imputation
crep, cpred = imputation.country_model(ds)
pd.DataFrame({k: {"mean": v[0], "sd": v[1]} for k, v in crep["accuracy"].items()}).T.assign(
    macro_f1=[crep["macro_f1"]["model"], None, crep["macro_f1"]["baseline_rule"]])

,mean,sd,macro_f1
model,0.884108,0.008936,0.861826
baseline_majority,0.313495,0.000274,NaN
baseline_rule,0.701525,0.007668,0.687147


In [3]:
print(f"top-3 accuracy: {crep['top3_accuracy_model']:.3f}")
cpred.sort_values("p_1", ascending=False).head(15)

top-3 accuracy: 0.969


,qid,name,pred_1,p_1,pred_2,p_2
394,Q7702513,Terkel Rosenqvist,NOR,0.999917,ISR,0.000032
238,Q6215598,Johann August Natterer,AUT,0.999913,RUS,0.000029
1007,Q28325135,Rajinder Kumar,IND,0.999847,USA,0.000048
407,Q7932730,Vinod K Singh,IND,0.999824,USA,0.000038
960,Q27929678,Murali Sastry,IND,0.999817,USA,0.000049
946,Q27899190,Paramasivam Natarajan,IND,0.999757,USA,0.000086
1015,Q28549858,Viswanathan Kumaran,IND,0.999757,USA,0.000135
1538,Q57913483,A. Harvey Millar,AUS,0.999685,OTHER,0.000074
184,Q5584302,Gopinath Kartha,IND,0.999608,RUS,0.000095
194,Q5629268,Wilhelm Eitel,DEU,0.999601,AUT,0.000088


In [4]:
srep, spred = imputation.specialty_model(ds)
{k: srep[k] for k in ("n", "accuracy", "macro_f1", "unweighted_model", "top3_accuracy_model")}

{'n': 4506,
 'accuracy': {'model': [0.6435835029321941, 0.011959859630715223],
  'baseline_majority': [0.6964048322755451, 0.000503939613348547],
  'baseline_rule': [0.6964048322755451, 0.000503939613348547]},
 'macro_f1': {'model': 0.33882497654710036,
  'baseline_rule': 0.13683933826518127},
 'unweighted_model': {'accuracy': [0.7137137597791072, 0.009270873614089792],
  'macro_f1': 0.30818906739989665},
 'top3_accuracy_model': 0.9347539442501679}

The specialty model doesn't beat the majority class on accuracy. With balanced weights it trades accuracy for macro-F1. Without occupation text, specialty can't be recovered from names, eras and institutions, so **we don't impute specialty**. That's the honest negative result.

In [5]:
from chemkg import genealogy
g, adv, _ = genealogy.build_graph(ds)
an = imputation.date_anomalies(ds, adv if len(adv) else None)
print(an.check.value_counts())
an[an.check == "advisor_age_gap"].head(15)

check
advisor_age_gap            682
conflicting_birth_dates    192
conflicting_death_dates    133
lifespan_under_15            3
lifespan_over_110            3
Name: count, dtype: int64


,qid,name,check,detail
331,Q26322,Carol Greider,advisor_age_gap,"as student: advisor b.1948, student b.1961 (gap 13)"
332,Q26321,Elizabeth Blackburn,advisor_age_gap,"as advisor: advisor b.1948, student b.1961 (gap 13)"
333,Q265310,Mostafa El-Sayed,advisor_age_gap,"as student: advisor b.1920, student b.1933 (gap 13)"
334,Q6831777,Michael Kasha,advisor_age_gap,"as advisor: advisor b.1920, student b.1933 (gap 13)"
335,Q271766,Gerald Edelman,advisor_age_gap,"as student: advisor b.1918, student b.1929 (gap 11)"
336,Q151564,Frederick Sanger,advisor_age_gap,"as advisor: advisor b.1918, student b.1929 (gap 11)"
337,Q273279,Roald Hoffmann,advisor_age_gap,"as student: advisor b.1931, student b.1937 (gap 6)"
338,Q102077243,Martin Gouterman,advisor_age_gap,"as advisor: advisor b.1931, student b.1937 (gap 6)"
339,Q26237589,Gene Brown,advisor_age_gap,"as student: advisor b.1914, student b.1926 (gap 12)"
340,Q27708663,Esmond Emerson Snell,advisor_age_gap,"as advisor: advisor b.1914, student b.1926 (gap 12)"


### Reading the result
- Names carry a lot of nationality signal. The country model reaches about 88% accuracy, against 70% for the birthplace rule and 31% for the majority class, with about 97% top-3 accuracy. That makes it useful for *suggesting* Wikidata edits, not for silently filling gaps.
- Advisor age gaps outside 15–80 years catch real errors: swapped advisor/student statements, and confused namesakes or birth years.